In [7]:
from __future__ import annotations
from datetime import datetime, timezone
from dateutil.relativedelta import relativedelta
from dateutil.tz import gettz
import json
from pathlib import Path

from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build


In [8]:
SCOPES = ["https://www.googleapis.com/auth/calendar.events.readonly"]

CREDS_JSON = "credentials.json"   # downloaded from Google Cloud Console
TOKEN_JSON = "token.json"         # generated after first login

def month_bounds(year: int, month: int, tz_name: str):
    tz = gettz(tz_name)
    start = datetime(year, month, 1, 0, 0, 0, tzinfo=tz)
    end = start + relativedelta(months=1)
    return start, end

def load_creds() -> Credentials:
    creds = None
    if Path(TOKEN_JSON).exists():
        creds = Credentials.from_authorized_user_file(TOKEN_JSON, SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CREDS_JSON, SCOPES)
            # prompt="consent" can help if you’re not receiving a refresh token on first run
            creds = flow.run_local_server(port=0, access_type="offline", prompt="consent")
        Path(TOKEN_JSON).write_text(creds.to_json())

    return creds

In [5]:
def upsert_row(conn, calendar_id, event_id, title, typ, date, duration_minutes, updated):
    conn.execute(
        """
        INSERT INTO calendar_events (calendar_id, event_id, title, type, date, duration_minutes, updated)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(calendar_id, event_id) DO UPDATE SET
          title = excluded.title,
          type = excluded.type,
          date = excluded.date,
          duration_minutes = excluded.duration_minutes,
          updated = excluded.updated
        """,
        (calendar_id, event_id, title, typ, date, duration_minutes, updated),
    )


In [6]:
def sync_last_month(service, conn: sqlite3.Connection, calendar_id: str = "https://calendar.google.com/calendar/u/0/r", days: int = 30):
    now = datetime.now(timezone.utc)
    time_min = (now - timedelta(days=days)).isoformat().replace("+00:00", "Z")
    time_max = now.isoformat().replace("+00:00", "Z")

    # Replace the stored window each run (simplest + avoids syncToken constraints)
    conn.execute("DELETE FROM calendar_events WHERE calendar_id = ?", (calendar_id,))

    page_token = None
    while True:
        resp = service.events().list(
            calendarId=calendar_id,
            singleEvents=True,    # expand recurring events into instances
            showDeleted=True,     # include cancelled events
            timeMin=time_min,
            timeMax=time_max,
            maxResults=2500,
            pageToken=page_token
        ).execute()

        for e in resp.get("items", []):
            title, typ, date, duration, updated = normalize_event(e)
            upsert_row(conn, calendar_id, e["id"], title, typ, date, duration, updated)

        page_token = resp.get("nextPageToken")
        if not page_token:
            break

    conn.commit()